# HighRes-Net Inference Diagnostics (HR-Expected)

Use this notebook when each input sample has both:
- 7 LR views (LR_*.tif/tiff)
- 1 HR counterpart (HR.tif/tiff)

This notebook focuses on:
- Low-VRAM tiled 2x inference (8 GB-friendly)
- Single-sample visual inspection
- HR-based metrics (PSNR, SSIM, RMSE, LPIPS)
- Optional multi-image benchmark and CSV report

## 1. Setup and Imports

In [ ]:
import sys
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import ndimage
import lpips

sys.path.insert(0, '../src')

import torch
from torch.utils.data import DataLoader
from DataLoader import TiffPatchDataset, collateFunction
from DeepNetworks.HRNet import HRNet
from diagnostics import (
    check_model_weights,
    compute_metrics,
 )

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
lpips_fn = lpips.LPIPS(net='alex').to(device)
lpips_fn.eval()
print(f"LPIPS metric initialized on {device}")

## 2. Quick Diagnostic: Check Image Loading First

In [ ]:
dataset_root = Path("D:\\GUC\\Datasets\\HighRes input test")

image_dirs = sorted([
    d for d in dataset_root.iterdir()
    if d.is_dir()
    and any(d.glob('LR_*.tif*'))
    and any(d.glob('HR.tif*'))
])

print("\n" + "=" * 70)
print("DATASET VERIFICATION (TIFF Format)")
print("=" * 70)
print(f"\nDataset root: {dataset_root}")
print(f"Found {len(image_dirs)} image folders")

diag_results = {}
if len(image_dirs) > 0:
    diag_results['num_images'] = len(image_dirs)
    print(f"✓ Image folders: {[d.name for d in image_dirs[:5]]}{'...' if len(image_dirs) > 5 else ''}")

    first_image = image_dirs[0]
    print(f"\nChecking first folder: {first_image.name}")

    lr_files = sorted(first_image.glob("LR_*.*"))
    hr_files = sorted(first_image.glob("HR.*"))

    print(f"  LR frames found: {len(lr_files)}")
    print(f"  HR files found:  {len(hr_files)}")

    if len(lr_files) > 0 and len(hr_files) > 0:
        print(f"  ✓ Structure looks good (LR frames + HR present)")
        diag_results['structure_valid'] = True
        lr_ext = lr_files[0].suffix
        hr_ext = hr_files[0].suffix
        print(f"  File types: LR{lr_ext}, HR{hr_ext}")
    else:
        print(f"  ✗ Missing LR frames or HR file!")
        diag_results['structure_valid'] = False
else:
    print(f"✗ No image folders found!")
    diag_results['num_images'] = 0

print("=" * 70)

## 3. Check Model Weights Status

In [ ]:
active_selection_path = Path("../models/weights/active_weight_selection.json")
legacy_weights_path   = Path("../models/weights/HRNet.pth")
resolved_weights_path = None

if active_selection_path.exists():
    try:
        with open(active_selection_path, 'r') as f:
            active_payload = json.load(f)
        selected_path = Path(active_payload.get('path', ''))
        if selected_path.exists():
            resolved_weights_path = selected_path
            print(f"Using active selected weight: {resolved_weights_path}")
        else:
            print(f"Active selection path not found on disk: {selected_path}")
    except Exception as e:
        print(f"Could not parse active selection file: {e}")

if resolved_weights_path is None and legacy_weights_path.exists():
    resolved_weights_path = legacy_weights_path
    print(f"Using legacy fallback weight: {resolved_weights_path}")

if resolved_weights_path is None:
    print("\n" + "=" * 70)
    print("⚠️  CRITICAL: No selected weight and no legacy HRNet.pth found")
    print("=" * 70)
    print("Use notebooks/weights_manager.ipynb to set an active weight selection.")
    weight_status = {'has_weights': False}
else:
    weight_status = check_model_weights(resolved_weights_path)

if not weight_status.get('has_weights', False):
    print("\n" + "=" * 70)
    print("⚠️  CRITICAL: Model is using RANDOM weights")
    print("=" * 70)
    print("""
This explains why your SR output looks bad:
- Grainy/noisy (random noise from uninitialized weights)
- Darker than expected (random negative scales)
- Artifacts/scan lines (random filter patterns)
NEXT STEPS:
1. Train the model on your data
2. Or find pre-trained weights for your domain
3. Or use bicubic upsampling for baseline comparison
    """)

## 4. Load Configuration

In [ ]:
config_path = '../config/config.json'
with open(config_path, 'r') as f:
    config = json.load(f)

network_cfg = config.get('network', {})
decoder_cfg = network_cfg.get('decoder', {})

if 'deconv' in decoder_cfg:
    upsample_desc = f"deconv stride={decoder_cfg['deconv'].get('stride', 'N/A')}x"
elif 'upsample' in decoder_cfg:
    up_cfg = decoder_cfg['upsample']
    upsample_desc = f"{up_cfg.get('mode', 'unknown')} scale_factor={up_cfg.get('scale_factor', 'N/A')}x"
else:
    upsample_desc = 'unknown (no deconv/upsample block found)'

print("Configuration:")
print(f"  n_views:         {config['training']['n_views']}")
print(f"  min_L:           {config['training']['min_L']}")
print(f"  batch_size:      {config['training']['batch_size']}")
print(f"  Decoder:         {upsample_desc}")
print(f"  Lambda range:    {config['training'].get('lambda_range', 'Not set')}")

## 5. Load Model Architecture

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

network_cfg = config.get('network', {})
weight_path_for_arch = resolved_weights_path if 'resolved_weights_path' in globals() else None

if weight_path_for_arch is not None:
    path_text = str(weight_path_for_arch).lower()
    if '\\base\\' in path_text:
        network_cfg['use_cbam'] = False
        print('Architecture override from weight path: Base -> use_cbam=False')
    elif '\\cbam\\' in path_text:
        network_cfg['use_cbam'] = True
        print('Architecture override from weight path: CBAM -> use_cbam=True')
    else:
        print(f"Architecture from config: use_cbam={network_cfg.get('use_cbam', False)}")
else:
    print(f"Architecture from config: use_cbam={network_cfg.get('use_cbam', False)}")

config['network']['use_cbam'] = bool(network_cfg.get('use_cbam', False))

model = HRNet(config['network'])
model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f"Model total parameters: {total_params:,}")
print(f"Model is in EVAL mode")
print(f"Effective use_cbam: {config['network']['use_cbam']}")

## 6. Load Pre-trained Weights

In [ ]:
active_selection_path = Path("../models/weights/active_weight_selection.json")
legacy_weights_path   = Path("../models/weights/HRNet.pth")
weights_path = None

if active_selection_path.exists():
    try:
        with open(active_selection_path, "r") as f:
            payload = json.load(f)
        raw_path = payload.get("path", "")
        if raw_path:
            candidate = Path(raw_path)
            if candidate.exists():
                weights_path = candidate
                print(f"Active selection: {weights_path}")
            else:
                print(f"WARNING: missing file: {candidate}")
    except Exception as e:
        print(f"Could not parse active_weight_selection.json: {e}")

if weights_path is None and legacy_weights_path.exists():
    weights_path = legacy_weights_path
    print(f"Falling back to legacy: {weights_path}")

if weights_path is None:
    raise FileNotFoundError("No weight found. Set one in weights_manager.ipynb.")

print(f"\nLoading: {weights_path}")
raw_ckpt = torch.load(str(weights_path), map_location=device, weights_only=False)

if isinstance(raw_ckpt, dict) and "state_dict" in raw_ckpt:
    state_dict = raw_ckpt["state_dict"]
elif isinstance(raw_ckpt, dict) and "model_state_dict" in raw_ckpt:
    state_dict = raw_ckpt["model_state_dict"]
else:
    state_dict = raw_ckpt

if isinstance(state_dict, dict):
    state_dict = {
        (k[7:] if isinstance(k, str) and k.startswith("module.") else k): v
        for k, v in state_dict.items()
    }
else:
    raise RuntimeError("Checkpoint format is not a valid state_dict dictionary.")

model_state = model.state_dict()
compatible, skipped = {}, []
for key, tensor in state_dict.items():
    if key not in model_state:
        skipped.append(f"{key}: not in model")
    elif tensor.shape != model_state[key].shape:
        skipped.append(f"{key}: shape {tuple(tensor.shape)} != {tuple(model_state[key].shape)}")
    else:
        compatible[key] = tensor

load_result = model.load_state_dict(compatible, strict=False)
missing_keys    = list(load_result.missing_keys)
unexpected_keys = list(load_result.unexpected_keys)

print(f"Keys in checkpoint: {len(state_dict)}")
print(f"Keys loaded:        {len(compatible)}")
print(f"Missing in model:   {len(missing_keys)}")
print(f"Unexpected in ckpt: {len(unexpected_keys)}")
print(f"Skipped by filter:  {len(skipped)}")

# Critical compatibility checks
critical_decoder_missing = [k for k in missing_keys if k.startswith("decode.")]
if critical_decoder_missing:
    print("\nCritical decoder keys missing (first 8 shown):")
    for k in critical_decoder_missing[:8]:
        print(f"  {k}")
    raise RuntimeError(
        "Decoder architecture mismatch between checkpoint and current model. "
        "Use a weight file trained with the current decoder schema."
    )

fa_ckpt_keys = [k for k in state_dict if k.startswith("fuse.fusion_attn.")]
fa_missing    = [k for k in missing_keys if k.startswith("fuse.fusion_attn.")]
if fa_ckpt_keys and fa_missing:
    print("\nFusionAttention mismatch (first 8 shown):")
    for k in fa_missing[:8]:
        print(f"  {k}")
    raise RuntimeError("FusionAttention keys exist in checkpoint but could not be loaded.")

legacy_no_fusion_attn = len(fa_ckpt_keys) == 0
if legacy_no_fusion_attn:
    print("\nNote: checkpoint has no FusionAttention keys (legacy checkpoint).")
    print("Switching to legacy blind fusion path for compatibility.")
    if hasattr(model, "fuse"):
        model.fuse.use_fusion_attention = False
else:
    if hasattr(model, "fuse"):
        model.fuse.use_fusion_attention = True

for name, module in model.named_modules():
    if hasattr(module, "blend") and hasattr(module, "query"):
        blend_val = torch.sigmoid(module.blend).item()
        print(f"  FusionAttention blend after load: sigmoid={blend_val:.4f}")

enc_std = model.encode.init_layer[0].weight.std().item()
print(f"\nLoad complete")
print(f"  Encoder std: {enc_std:.6f}  {'trained-like' if enc_std > 0.02 else 'possibly weak/random'}")
model.eval()

with torch.no_grad():
    dummy_lr    = torch.zeros(1, 7, 32, 32).to(device)
    dummy_alpha = torch.ones(1, 7).to(device)
    dummy_out   = model(dummy_lr, dummy_alpha)
    out_mean    = dummy_out.mean().item()
    print(f"  Forward sanity: output mean = {out_mean:.4f}")

In [ ]:
# Ablation controls (inference only)
ABLATION_MODE = "both"  # options: both, no_cbam, no_fusion, no_both

def _set_cbam_enabled(model_instance, enabled):
    for module in model_instance.modules():
        if module.__class__.__name__ == "CBAM":
            if enabled:
                if hasattr(module, "_orig_forward"):
                    module.forward = module._orig_forward
            else:
                if not hasattr(module, "_orig_forward"):
                    module._orig_forward = module.forward
                module.forward = lambda x: x

def _set_fusion_attention_enabled(model_instance, enabled):
    if hasattr(model_instance, "fuse") and hasattr(model_instance.fuse, "use_fusion_attention"):
        model_instance.fuse.use_fusion_attention = bool(enabled)

if ABLATION_MODE not in ("both", "no_cbam", "no_fusion", "no_both"):
    raise ValueError("ABLATION_MODE must be: both, no_cbam, no_fusion, no_both")

cbam_enabled = ABLATION_MODE in ("both", "no_fusion")
fusion_enabled = ABLATION_MODE in ("both", "no_cbam")

_set_cbam_enabled(model, cbam_enabled)
_set_fusion_attention_enabled(model, fusion_enabled)
print(f"Ablation mode: {ABLATION_MODE} | cbam={cbam_enabled} fusion={fusion_enabled}")

## 7. Create Dataset and DataLoader

In [ ]:
dataset_root = Path("D:\\GUC\\Datasets\\HighRes input test")

image_dirs = sorted([
    d for d in dataset_root.iterdir()
    if d.is_dir()
    and any(d.glob('LR_*.tif*'))
    and any(d.glob('HR.*'))
])
print(f"Found {len(image_dirs)} images")
if len(image_dirs) > 0:
    print(f"Images: {[d.name for d in image_dirs[:5]]}...")

image_dirs_str = [str(d) for d in image_dirs]

dataset = TiffPatchDataset(
    patch_dirs=image_dirs_str,
    max_views=config['training']['n_views']
 )
print(f"Dataset size: {len(dataset)} images")

min_L = config['training']['min_L']
inference_batch_size = 1
n_workers = int(config['training'].get('n_workers', 0))
pin_memory = torch.cuda.is_available()
force_single_worker = os.name == 'nt'
if force_single_worker and n_workers > 0:
    print('Windows notebook: forcing num_workers=0 to avoid worker crashes.')
    n_workers = 0
use_workers = n_workers > 0
loader_kwargs = {
    'num_workers': n_workers,
    'pin_memory': pin_memory,
}
if use_workers:
    loader_kwargs.update({
        'persistent_workers': True,
        'prefetch_factor': 2,
    })
dataloader = DataLoader(
    dataset,
    batch_size=inference_batch_size,
    shuffle=False,
    collate_fn=collateFunction(min_L=min_L),
    **loader_kwargs,
 )
print(f"DataLoader created with batch_size={inference_batch_size}")
print(f"  num_workers={n_workers}  pin_memory={pin_memory}")
if use_workers:
    print("  persistent_workers=True  prefetch_factor=2")

## 7. Create Dataset and DataLoader

In [ ]:
batch = next(iter(dataloader))
lrs, alphas, hrs, hr_maps, names = batch

print(f"Batch loaded:")
print(f"  LR shape:   {lrs.shape}")
print(f"  Alphas:     {alphas.shape}")
print(f"  HR shape:   {hrs.shape}")
print(f"  HR_map:     {hr_maps.shape}")
print(f"  Names:      {names}")

lrs    = lrs.float().cpu()
alphas = alphas.float().cpu()
print("\nData kept on CPU for tiled low-VRAM inference")

## 8. Run Inference on First Batch

In [ ]:
batch = next(iter(dataloader))
lrs, alphas, hrs, hr_maps, names = batch

print(f"Batch loaded:")
print(f"  LR shape:   {lrs.shape}")
print(f"  Alphas:     {alphas.shape}")
print(f"  HR shape:   {hrs.shape}")
print(f"  HR_map:     {hr_maps.shape}")
print(f"  Names:      {names}")

lrs    = lrs.float().cpu()
alphas = alphas.float().cpu()
print("\nData kept on CPU for tiled low-VRAM inference")

## 9. Tiled Forward Pass (Single Safe 8 GB Profile)

In [ ]:
inference_cfg = {
    'tile_size': 160,
    'halo': 96,
    'use_amp': False,
    'blend_power': 1.5,
}
print(f"Inference config: {inference_cfg}")


def _blend_window_1d(size, device, dtype):
    if size <= 1:
        return torch.ones(size, device=device, dtype=dtype)
    window = torch.hann_window(size, periodic=False, device=device, dtype=dtype)
    return window.clamp_min(1e-3)


def _blend_weight_2d(height, width, device, dtype, power=1.0, min_weight=1e-3):
    wy = _blend_window_1d(height, device, dtype)
    wx = _blend_window_1d(width, device, dtype)
    weight = torch.outer(wy, wx)
    if power != 1.0:
        weight = weight.pow(power)
    weight = weight.clamp_min(min_weight)
    return weight / weight.max().clamp_min(min_weight)


def tiled_forward_hrnet(model, lrs_cpu, alphas_cpu, device,
                        tile_size=192, halo=96, use_amp=False, blend_power=1.5):
    """
    Memory-safe tiled HRNet inference with weighted blending to hide seams.
    """
    model.eval()
    outputs = []
    with torch.no_grad():
        B, _, H, W = lrs_cpu.shape
        step        = int(tile_size)
        hctx        = int(halo)
        amp_enabled = bool(use_amp and device.type == 'cuda')

        # Probe to infer integer upscale factor
        probe_h  = min(64, H)
        probe_w  = min(64, W)
        probe_lr = lrs_cpu[:1, :, :probe_h, :probe_w].to(device, non_blocking=True)
        probe_al = alphas_cpu[:1].to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
            probe_sr = model(probe_lr, probe_al)[:, 0]
        sy = max(1, int(round(probe_sr.shape[-2] / probe_h)))
        sx = max(1, int(round(probe_sr.shape[-1] / probe_w)))
        del probe_lr, probe_al, probe_sr

        crop_h = step * sy
        crop_w = step * sx
        blend_weight = _blend_weight_2d(
            crop_h, crop_w, device='cpu', dtype=torch.float32, power=blend_power
        ).unsqueeze(0)

        for b in range(B):
            sample = lrs_cpu[b:b+1]
            alpha  = alphas_cpu[b:b+1]

            Hp = ((H + step - 1) // step) * step
            Wp = ((W + step - 1) // step) * step
            sample_grid = torch.nn.functional.pad(
                sample, (0, Wp - W, 0, Hp - H), mode='reflect')
            sample_ext  = torch.nn.functional.pad(
                sample_grid, (hctx, hctx, hctx, hctx), mode='reflect')

            out_hp = Hp * sy
            out_wp = Wp * sx
            sr_acc = torch.zeros((1, out_hp, out_wp), dtype=torch.float32)
            sr_wgt = torch.zeros((1, out_hp, out_wp), dtype=torch.float32)

            for y0 in range(0, Hp, step):
                for x0 in range(0, Wp, step):
                    lr_tile  = sample_ext[:, :,
                                         y0:y0 + step + 2 * hctx,
                                         x0:x0 + step + 2 * hctx].to(device, non_blocking=True)
                    al_tile  = alpha.to(device, non_blocking=True)
                    with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=amp_enabled):
                        sr_tile = model(lr_tile, al_tile)[:, 0]
                    sr_tile = sr_tile.float().cpu()

                    cy0  = hctx * sy
                    cy1  = cy0 + step * sy
                    cx0  = hctx * sx
                    cx1  = cx0 + step * sx
                    crop = sr_tile[:, cy0:cy1, cx0:cx1]

                    ty0 = y0 * sy
                    ty1 = ty0 + step * sy
                    tx0 = x0 * sx
                    tx1 = tx0 + step * sx
                    sr_acc[:, ty0:ty1, tx0:tx1] += crop * blend_weight
                    sr_wgt[:, ty0:ty1, tx0:tx1] += blend_weight
                    del lr_tile, al_tile, sr_tile, crop

            sr_grid = sr_acc / sr_wgt.clamp_min(1e-6)
            sr_out  = sr_grid[:, :H * sy, :W * sx].unsqueeze(0)
            outputs.append(sr_out)

    return torch.cat(outputs, dim=0)


if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    print(f"GPU memory before: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

try:
    sr_output = tiled_forward_hrnet(
        model=model,
        lrs_cpu=lrs,
        alphas_cpu=alphas,
        device=device,
        tile_size=inference_cfg['tile_size'],
        halo=inference_cfg['halo'],
        use_amp=inference_cfg['use_amp'],
        blend_power=inference_cfg['blend_power'],
    )
except torch.cuda.OutOfMemoryError:
    print("\n⚠️  CUDA OOM. Retrying with safer settings...")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    sr_output = tiled_forward_hrnet(
        model=model, lrs_cpu=lrs, alphas_cpu=alphas,
        device=device, tile_size=128, halo=96, use_amp=False, blend_power=1.5,
    )

if torch.cuda.is_available():
    print(f"GPU memory after: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    print(f"Peak GPU memory:  {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")

print(f"\nSR output shape: {sr_output.shape}")
print(f"SR value range:  [{sr_output.min():.4f}, {sr_output.max():.4f}]")

## 10. Extract and Analyze First Sample

In [ ]:
sr       = np.clip(sr_output[0, 0].cpu().numpy(), 0, 1)
lr_first = lrs[0, 0].cpu().numpy()
hr_true  = hrs[0].numpy() if torch.is_tensor(hrs) else hrs[0] if isinstance(hrs, list) else None

if hr_true is not None and hr_true.ndim == 3:
    hr_true = np.squeeze(hr_true, axis=0)

lr_upsampled = np.repeat(np.repeat(lr_first, 2, axis=0), 2, axis=1)

print(f"Extracted first sample:")
print(f"  LR:  {lr_first.shape}")
print(f"  SR:  {sr.shape}")
print(f"  HR:  {hr_true.shape if hr_true is not None else 'Not loaded'}")

## 11. Compute Quality Metrics

In [ ]:
print("\n" + "=" * 70)
print("Image Quality Metrics (SR vs Ground Truth)")
print("=" * 70)

if hr_true is not None:
    if hr_true.ndim == 3:
        hr_true = np.squeeze(hr_true, axis=0)

    hr_map_true = None
    if 'hr_maps' in globals() and hr_maps is not None:
        hr_map_true = (hr_maps[0].cpu().numpy()
                       if torch.is_tensor(hr_maps) else hr_maps[0])

    if hr_true.shape != sr.shape:
        scale   = sr.shape[0] / hr_true.shape[0]
        hr_true = ndimage.zoom(hr_true, scale, order=3)
        if hr_map_true is not None:
            hr_map_true = ndimage.zoom(hr_map_true, scale, order=0)

    nn_metrics = compute_metrics(hr_true, sr, hr_map=hr_map_true)

    if nn_metrics['valid']:
        print(f"\n🧠 HighRes-Net Output (RAW metrics):")
        print(f"  PSNR:               {nn_metrics['psnr']:.2f} dB  ({nn_metrics['psnr_label']})")
        print(f"  SSIM:               {nn_metrics['ssim']:.4f}  ({nn_metrics['ssim_label']})")
        print(f"  RMSE:               {nn_metrics['rmse']:.5f}")
        print(f"  Edge ratio (SR/HR): {nn_metrics['edge_ratio']:.3f}  (near 1 is ideal)")
        print(f"  Contrast ratio:     {nn_metrics['contrast_ratio']:.3f}  (near 1 is ideal)")
        print(f"  P99-P1 range ratio: {nn_metrics['p01_p99_range_ratio']:.3f}  (near 1 is ideal)")
        print(f"\n🔎 Brightness-corrected view (diagnostic only):")
        print(f"  Brightness bias (HR-SR):    {nn_metrics['brightness_bias']:+.5f}")
        print(f"  PSNR (bias-corrected):      {nn_metrics['psnr_bias_corrected']:.2f} dB")
        print(f"  SSIM (bias-corrected):      {nn_metrics['ssim_bias_corrected']:.4f}")

        try:
            sr_t  = torch.from_numpy(sr).float().unsqueeze(0).unsqueeze(0).to(device)
            hr_t  = torch.from_numpy(hr_true).float().unsqueeze(0).unsqueeze(0).to(device)
            sr_t  = sr_t * 2 - 1
            hr_t  = hr_t * 2 - 1
            with torch.no_grad():
                lpips_score = lpips_fn(sr_t, hr_t).item()
            lpips_label = (
                "Excellent (< 0.1)"   if lpips_score < 0.10 else
                "Very Good (0.1-0.15)" if lpips_score < 0.15 else
                "Good (0.15-0.20)"    if lpips_score < 0.20 else
                "Acceptable (0.20-0.30)" if lpips_score < 0.30 else
                "Poor (> 0.30)"
            )
            print(f"\n  LPIPS: {lpips_score:.4f} ({lpips_label})")
            print("  Note: LPIPS is useful as a trend signal, not a single source of truth.")
        except Exception as e:
            print(f"  ⚠️  LPIPS computation failed: {e}")

    # Bicubic baseline
    lr_bicubic = ndimage.zoom(lr_first, 2, order=3)
    if lr_bicubic.shape != hr_true.shape:
        scale      = hr_true.shape[0] / lr_bicubic.shape[0]
        lr_bicubic = ndimage.zoom(lr_bicubic, scale, order=3)

    bicubic_metrics = compute_metrics(hr_true, lr_bicubic, hr_map=hr_map_true)

    if bicubic_metrics['valid']:
        print(f"\n📊 Bicubic Baseline (RAW metrics):")
        print(f"  PSNR:               {bicubic_metrics['psnr']:.2f} dB  ({bicubic_metrics['psnr_label']})")
        print(f"  SSIM:               {bicubic_metrics['ssim']:.4f}  ({bicubic_metrics['ssim_label']})")
        print(f"  RMSE:               {bicubic_metrics['rmse']:.5f}")
        print(f"  Edge ratio:         {bicubic_metrics['edge_ratio']:.3f}")
        print(f"  Contrast ratio:     {bicubic_metrics['contrast_ratio']:.3f}")

        try:
            bc_t = torch.from_numpy(lr_bicubic).float().unsqueeze(0).unsqueeze(0).to(device)
            hr_t = torch.from_numpy(hr_true).float().unsqueeze(0).unsqueeze(0).to(device)
            bc_t = bc_t * 2 - 1
            hr_t = hr_t * 2 - 1
            with torch.no_grad():
                lpips_bicubic = lpips_fn(bc_t, hr_t).item()
            lpips_bc_label = (
                "Excellent (< 0.1)"      if lpips_bicubic < 0.10 else
                "Very Good (0.1-0.15)"   if lpips_bicubic < 0.15 else
                "Good (0.15-0.20)"       if lpips_bicubic < 0.20 else
                "Acceptable (0.20-0.30)" if lpips_bicubic < 0.30 else
                "Poor (> 0.30)"
            )
            print(f"  LPIPS: {lpips_bicubic:.4f} ({lpips_bc_label})")
        except Exception as e:
            print(f"  ⚠️  LPIPS computation failed: {e}")

        if nn_metrics['valid']:
            psnr_diff = nn_metrics['psnr']         - bicubic_metrics['psnr']
            ssim_diff = nn_metrics['ssim']         - bicubic_metrics['ssim']
            edge_diff = nn_metrics['edge_ratio']   - bicubic_metrics['edge_ratio']
            print(f"\n📈 HRNet vs Bicubic: PSNR {psnr_diff:+.2f} dB  "
                  f"SSIM {ssim_diff:+.4f}  Edge ratio {edge_diff:+.3f}")
            print("  Better model should improve several metrics together, not PSNR alone.")
else:
    print("HR ground truth not available")

In [ ]:
print("\n" + "=" * 70)
print("OUTPUT RANGE ANALYSIS")
print("=" * 70)

sr_batch_min   = sr_output.min().item()
sr_batch_max   = sr_output.max().item()
sr_batch_range = sr_batch_max - sr_batch_min
hrs_batch_min  = hrs.min().item()
hrs_batch_max  = hrs.max().item()
hrs_batch_range = hrs_batch_max - hrs_batch_min

print(f"\n📊 SR Output:  min={sr_batch_min:.4f}  max={sr_batch_max:.4f}  "
      f"range={sr_batch_range:.4f}  mean={sr_output.mean():.4f}  std={sr_output.std():.4f}")
print(f"📊 HR Target:  min={hrs_batch_min:.4f}  max={hrs_batch_max:.4f}  "
      f"range={hrs_batch_range:.4f}  mean={hrs.mean():.4f}  std={hrs.std():.4f}")

if sr_batch_range < 0.3:
    print(f"\n🔴 RANGE SEVERELY COMPRESSED ({sr_batch_range:.4f} < 0.3) → likely blur/flat contrast")
elif sr_batch_range < hrs_batch_range * 0.7:
    print(f"\n🟠 RANGE COMPRESSED: SR {sr_batch_range:.4f} vs HR {hrs_batch_range:.4f}")
elif sr_batch_range > hrs_batch_range * 1.3:
    print(f"\n🟠 RANGE TOO WIDE: SR {sr_batch_range:.4f} vs HR {hrs_batch_range:.4f}")
else:
    print(f"\n✓ RANGE GOOD: SR {sr_batch_range:.4f} matches HR {hrs_batch_range:.4f}")

sr_std    = sr_output.std().item()
hrs_std   = hrs.std().item()
std_ratio = sr_std / hrs_std if hrs_std > 0 else 0
print(f"\n📈 Contrast (std): SR/HR = {std_ratio:.3f}", end="  ")
if std_ratio < 0.7:
    print("⚠️  Less contrast than HR (blurry)")
elif std_ratio > 1.3:
    print("⚠️  More contrast than HR (over-enhanced)")
else:
    print("✓ Looks good")

print("=" * 70)

## 12. Visualize Comparison

In [ ]:
if hr_true is not None:
    fig, axes = plt.subplots(2, 2, figsize=(12, 12))

    axes[0, 0].imshow(lr_upsampled, cmap='gray', vmin=0, vmax=1)
    axes[0, 0].set_title('LR View (2x nearest-neighbor)')
    axes[0, 0].axis('off')

    lr_bicubic_disp = ndimage.zoom(lr_first, 2, order=3)
    axes[0, 1].imshow(lr_bicubic_disp, cmap='gray', vmin=0, vmax=1)
    bc_title = (f'LR (2x bicubic)\nPSNR = {bicubic_metrics["psnr"]:.2f} dB'
                if bicubic_metrics['valid'] else 'LR (2x bicubic)')
    axes[0, 1].set_title(bc_title)
    axes[0, 1].axis('off')

    axes[1, 0].imshow(sr, cmap='gray', vmin=0, vmax=1)
    sr_title = (f'HighRes-Net SR\nPSNR = {nn_metrics["psnr"]:.2f} dB'
                if nn_metrics['valid'] else 'HighRes-Net SR')
    axes[1, 0].set_title(sr_title)
    axes[1, 0].axis('off')
    if not weight_status.get('has_weights', False):
        axes[1, 0].text(0.5, 0.02, '(Random weights)',
                        ha='center', transform=axes[1, 0].transAxes,
                        fontsize=10, color='red', weight='bold')

    axes[1, 1].imshow(hr_true, cmap='gray', vmin=0, vmax=1)
    axes[1, 1].set_title('HR Ground Truth')
    axes[1, 1].axis('off')

    plt.tight_layout()
    plt.show()

    # Full-resolution isolated SR
    plt.figure(figsize=(12, 12), dpi=200)
    plt.imshow(sr, cmap='gray', vmin=0, vmax=1)
    plt.title('HighRes-Net SR (Full Resolution)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Full-resolution isolated HR
    plt.figure(figsize=(12, 12), dpi=200)
    plt.imshow(hr_true, cmap='gray', vmin=0, vmax=1)
    plt.title('HR Ground Truth (Full Resolution)')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    print(f"SR shape: {sr.shape}  HR shape: {hr_true.shape}")
else:
    print("Cannot visualize: HR not loaded")

# Ablation controls (inference only)
ABLATION_MODE = "both"  # options: both, no_cbam, no_fusion, no_both

def _set_cbam_enabled(model_instance, enabled):
    for module in model_instance.modules():
        if module.__class__.__name__ == "CBAM":
            if enabled:
                if hasattr(module, "_orig_forward"):
                    module.forward = module._orig_forward
                    delattr(module, "_orig_forward")
            else:
                if not hasattr(module, "_orig_forward"):
                    module._orig_forward = module.forward
                module.forward = lambda x, *args, **kwargs: x

def _set_fusion_attention_enabled(model_instance, enabled):
    # Keep FusionAttention path; suppress its blend contribution when disabled.
    for module in model_instance.modules():
        if hasattr(module, "blend") and hasattr(module, "query"):
            if enabled:
                if hasattr(module, "_orig_blend"):
                    with torch.no_grad():
                        module.blend.copy_(module._orig_blend)
                    delattr(module, "_orig_blend")
            else:
                if not hasattr(module, "_orig_blend"):
                    module._orig_blend = module.blend.detach().clone()
                with torch.no_grad():
                    module.blend.fill_(-10.0)

if ABLATION_MODE not in ("both", "no_cbam", "no_fusion", "no_both"):
    raise ValueError("ABLATION_MODE must be: both, no_cbam, no_fusion, no_both")

cbam_enabled = ABLATION_MODE in ("both", "no_fusion")
fusion_enabled = ABLATION_MODE in ("both", "no_cbam")

_set_cbam_enabled(model, cbam_enabled)
_set_fusion_attention_enabled(model, fusion_enabled)
print(f"Ablation mode: {ABLATION_MODE} | cbam={cbam_enabled} fusion={fusion_enabled}")

In [ ]:
import pandas as pd

def run_multi_image_benchmark(model, dataloader, device, lpips_fn, max_samples=250):
    rows = []
    model.eval()
    with torch.no_grad():
        processed = 0
        for batch_idx, batch in enumerate(dataloader):
            if processed >= max_samples:
                break
            lrs_b, alphas_b, hrs_b, hr_maps_b, names_b = batch

            lrs_cpu    = lrs_b.float().cpu()
            alphas_cpu = alphas_b.float().cpu()

            srs_b = tiled_forward_hrnet(
                model=model, lrs_cpu=lrs_cpu, alphas_cpu=alphas_cpu,
                device=device,
                tile_size=inference_cfg['tile_size'],
                halo=inference_cfg['halo'],
                use_amp=inference_cfg['use_amp'],
            )[:, 0].detach().cpu().numpy()

            lrs_np     = lrs_cpu[:, 0].detach().cpu().numpy()
            hrs_np     = (hrs_b.detach().cpu().numpy()
                          if torch.is_tensor(hrs_b) else np.asarray(hrs_b))
            hr_maps_np = (hr_maps_b.detach().cpu().numpy()
                          if torch.is_tensor(hr_maps_b) else np.asarray(hr_maps_b))

            for i in range(srs_b.shape[0]):
                if processed >= max_samples:
                    break

                sr_i  = np.clip(srs_b[i], 0, 1)
                hr_i  = np.clip(np.squeeze(hrs_np[i], axis=0)
                                if hrs_np[i].ndim == 3 else hrs_np[i], 0, 1)
                map_i = hr_maps_np[i]

                if hr_i.shape != sr_i.shape:
                    scale = sr_i.shape[0] / hr_i.shape[0]
                    hr_i  = ndimage.zoom(hr_i, scale, order=3)
                    map_i = ndimage.zoom(map_i, scale, order=0)

                lr_bicubic_i = ndimage.zoom(lrs_np[i], 2, order=3)
                if lr_bicubic_i.shape != hr_i.shape:
                    scale        = hr_i.shape[0] / lr_bicubic_i.shape[0]
                    lr_bicubic_i = ndimage.zoom(lr_bicubic_i, scale, order=3)

                m_sr = compute_metrics(hr_i, sr_i,          hr_map=map_i)
                m_bc = compute_metrics(hr_i, lr_bicubic_i,  hr_map=map_i)

                lpips_sr = lpips_bc = np.nan
                try:
                    sr_t = (torch.from_numpy(sr_i).float()
                            .unsqueeze(0).unsqueeze(0).to(device) * 2 - 1)
                    hr_t = (torch.from_numpy(hr_i).float()
                            .unsqueeze(0).unsqueeze(0).to(device) * 2 - 1)
                    bc_t = (torch.from_numpy(lr_bicubic_i).float()
                            .unsqueeze(0).unsqueeze(0).to(device) * 2 - 1)
                    lpips_sr = float(lpips_fn(sr_t, hr_t).item())
                    lpips_bc = float(lpips_fn(bc_t, hr_t).item())
                except Exception:
                    pass

                rows.append({
                    'name':                names_b[i] if isinstance(names_b, list) else f'image_{processed}',
                    'psnr_sr':             m_sr.get('psnr', np.nan),
                    'ssim_sr':             m_sr.get('ssim', np.nan),
                    'rmse_sr':             m_sr.get('rmse', np.nan),
                    'edge_ratio_sr':       m_sr.get('edge_ratio', np.nan),
                    'contrast_ratio_sr':   m_sr.get('contrast_ratio', np.nan),
                    'lpips_sr':            lpips_sr,
                    'psnr_bicubic':        m_bc.get('psnr', np.nan),
                    'ssim_bicubic':        m_bc.get('ssim', np.nan),
                    'rmse_bicubic':        m_bc.get('rmse', np.nan),
                    'edge_ratio_bicubic':  m_bc.get('edge_ratio', np.nan),
                    'contrast_ratio_bicubic': m_bc.get('contrast_ratio', np.nan),
                    'lpips_bicubic':       lpips_bc,
                    'delta_psnr':          m_sr.get('psnr', np.nan) - m_bc.get('psnr', np.nan),
                    'delta_ssim':          m_sr.get('ssim', np.nan) - m_bc.get('ssim', np.nan),
                    'delta_rmse':          m_sr.get('rmse', np.nan) - m_bc.get('rmse', np.nan),
                    'delta_lpips':         lpips_sr - lpips_bc,
                })
                processed += 1

    df = pd.DataFrame(rows)
    if len(df) == 0:
        print("No samples were evaluated.")
        return None

    summary = pd.Series({
        'samples':          len(df),
        'mean_delta_psnr':  df['delta_psnr'].mean(),
        'mean_delta_ssim':  df['delta_ssim'].mean(),
        'mean_delta_rmse':  df['delta_rmse'].mean(),
        'mean_delta_lpips': df['delta_lpips'].mean(),
        'hrnet_wins_psnr':  int((df['delta_psnr'] > 0).sum()),
        'hrnet_wins_ssim':  int((df['delta_ssim'] > 0).sum()),
        'hrnet_wins_lpips': int((df['delta_lpips'] < 0).sum()),
    })
    return df, summary


print("\n" + "=" * 70)
print("MULTI-IMAGE BENCHMARK (HRNet vs Bicubic)")
print("=" * 70)

benchmark_result = run_multi_image_benchmark(
    model=model,
    dataloader=dataloader,
    device=device,
    lpips_fn=lpips_fn,
    max_samples=250
)

if benchmark_result is not None:
    benchmark_df, benchmark_summary = benchmark_result
    print("\nSummary:")
    print(benchmark_summary)
    print("\nTop 10 by PSNR gain over bicubic:")
    display(benchmark_df.sort_values('delta_psnr', ascending=False).head(10))
    print("\nMedian deltas (robust overview):")
    display(benchmark_df[['delta_psnr', 'delta_ssim',
                           'delta_rmse', 'delta_lpips']].median().to_frame('median'))

    benchmark_csv = Path("../models/weights/checkpoints/benchmark_last_run.csv")
    benchmark_csv.parent.mkdir(parents=True, exist_ok=True)
    benchmark_df.to_csv(benchmark_csv, index=False)
    print(f"\nSaved benchmark to: {benchmark_csv}")

print("=" * 70)